In [87]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import bisect, brentq
from copy import copy

# Problem 1

In [88]:
class UpAndOutPut:

    def __init__(self, K, T, barrier, observationinterval):
        self.K = K
        self.T = T
        self.barrier = barrier
        self.observationinterval = observationinterval

In [89]:
hw2contract = UpAndOutPut(K=95, T=0.25, barrier=114, observationinterval=0.02)

In [90]:
class GBMdynamics:

    def __init__(self, S, r, rGrow, sigma=None):
        self.S = S
        self.r = r
        self.rGrow = rGrow
        self.sigma = sigma

    def update_sigma(self, sigma):
        self.sigma = sigma
        return self

In [91]:
# Use the same GBMdynamics class from HW 1

hw2dynamics = GBMdynamics(S=100, sigma=0.4, rGrow=0, r=0)

In [92]:
class TreeEngine:

    def __init__(self, N):
        self.N = N

    def price_upandout(self, dynamics, contract):

        deltat = contract.T / self.N
        J = np.ceil(np.log(contract.barrier/dynamics.S)/(dynamics.sigma*np.sqrt(3*deltat))-0.5)
        deltax = np.log(contract.barrier/dynamics.S)/(J+0.5)

        Sgrid = dynamics.S*np.exp(np.linspace(self.N, -self.N, num=2*self.N+1, endpoint=True)*deltax)
        #Here I decided to make the SMALLER indexes in this array correspond to HIGHER S

        numTimestepsPerObs = contract.observationinterval/deltat
        if abs(numTimestepsPerObs-round(numTimestepsPerObs)) > 1e-8:
            raise ValueError("This value of N fails to place the observation dates in the tree.")

        # complete this
        nu = dynamics.rGrow - 0.5 * dynamics.sigma**2       
        Pu = 0.5 * ((dynamics.sigma**2 * deltat + nu**2 * deltat**2) / (deltax**2) + nu * deltat / deltax)       # complete this
        Pd = 0.5 * ((dynamics.sigma**2 * deltat + nu**2 * deltat**2) / (deltax**2) - nu * deltat / deltax)       # complete this
        Pm = 1 - Pu - Pd # shortcut 

        optionprice = np.maximum(contract.K-Sgrid,0)   #an array of time-T option prices.
        # print(optionprice)

        #Next, induct backwards to time 0, updating the optionprice array
        #Hint: if x is an array, then what are x[2:] and x[1:-1] and x[:-2]

        for t in np.linspace(self.N-1, 0, num=self.N, endpoint=True)*deltat:
            
            # insert lines of code here if needed

            # print(t)

            Sgrid = Sgrid[1:-1]

            optionprice = np.exp(-dynamics.r*deltat)*(Pu*optionprice[0:-2] + Pm*optionprice[1:-1] + Pd*optionprice[2:])

            if abs(t % contract.observationinterval) < 1e-8 and t > 0:
                optionprice = np.where(Sgrid >= contract.barrier, 0, optionprice)

        return optionprice[0]
        #The [0] is assuming that we are shrinking the optionprice array in each iteration of the loop,
        #until finally there is only 1 element in the array.
        #If instead you are keeping unchanged the size of the optionprice array in each iteration,
        #then you need to change the [0] to a different index.


In [93]:
hw2tree=TreeEngine(N=100)

hw2tree.price_upandout(hw2dynamics, hw2contract)

np.float64(5.350527169431585)

# Problem 2

In [94]:
# Uses the CallOption and AnalyticEngine classes from HW 1

In [96]:
hw2analytic = AnalyticEngine()

hw2analytic.IV(GBMdynamics(sigma=None, rGrow=0, S=100, r=0),
               CallOption(K=100, T=0.5, price=12))

NameError: name 'AnalyticEngine' is not defined